# Pandas: Data Manipulation


## Table of contents

 1. <a href="#1.-Create-DataFrame">Create DataFrame</a>  
 2. <a href="#2.-Combining-DataFrames">Combining DataFrames</a>  
 3. <a href="#3.-Summarizing">Summarizing</a>  
 4. <a href="#4.-Columns-selection">Columns selection</a>  
 5. <a href="#5.-Rows-selection-(basic)">Rows selection (basic)</a>  
 6. <a href="#6.-Row-selection-(filtering)">Row selection (filtering)</a>  
 7. <a href="#7.-Sorting">Sorting</a>  
 8. <a href="#8.-Descriptive-statistics">Descriptive statistics</a>  
 9. <a href="#9.-Quality-check">Quality check</a>  
 10. <a href="#10.-Rename-values">Rename values</a>  
 11. <a href="#11.-File-I/O">File I/O</a>  

### Pandas가 지원하는 Data structure
* **Series** 는 value와 index의 형태를 지니는 1 차원 배열로서, 어떤 데이터 타입도 요소가 될 수 있다.

* **DataFrame** 은 2 차원 배열로 열(column)들 끼리는 다른 데이터 타입일 수도 있다. 엑셀 spreadsheet을 생각하면 된다.

In [1]:
from __future__ import print_function

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Create DataFrame

In [2]:
columns = ['name', 'age', 'gender', 'job']

user1 = pd.DataFrame([['alice', 19, "F", "student"],
                      ['john', 26, "M", "student"]],
        columns=columns)

user2 = pd.DataFrame([['eric', 22, "M", "student"],
                      ['paul', 58, "F", "manager"]],
        columns=columns)

user3 = pd.DataFrame(dict(name=['peter', 'julie'],
                          age=[33, 44], gender=['M', 'F'],
                          job=['engineer', 'scientist']))

In [3]:
# Pandas가 출력 form을 보기 좋게 맞춰준다
user1

,name,age,gender,job
0,alice,19,F,student
1,john,26,M,student


In [4]:
print(user1)

    name  age gender      job
0  alice   19      F  student
1   john   26      M  student


In [5]:
user2

,name,age,gender,job
0,eric,22,M,student
1,paul,58,F,manager


In [6]:
# 딕셔너리로 생성한 user3는 column 순서가 기대했던 것과는 다르다.
user3

,name,age,gender,job
0,peter,33,M,engineer
1,julie,44,F,scientist


[<a href="#Pandas:-Data-Manipulation">Back to top</a>]

## 2. Combining DataFrames

### Concatenate DataFrame

In [7]:
user1.append(user2)
user1

,name,age,gender,job
0,alice,19,F,student
1,john,26,M,student


In [8]:
users = pd.concat([user1, user2, user3])
users

,name,age,gender,job
0,alice,19,F,student
1,john,26,M,student
0,eric,22,M,student
1,paul,58,F,manager
0,peter,33,M,engineer
1,julie,44,F,scientist


### Join DataFrame

In [9]:
user4 = pd.DataFrame(dict(name=['alice', 'john', 'eric', 'julie'],
                          height=[165, 180, 175, 171]))
user4

,name,height
0,alice,165
1,john,180
2,eric,175
3,julie,171


In [10]:
# Use intersection of keys from both frames
merge_inter = pd.merge(users, user4, on="name")
merge_inter

,name,age,gender,job,height
0,alice,19,F,student,165
1,john,26,M,student,180
2,eric,22,M,student,175
3,julie,44,F,scientist,171


In [11]:
# Use union of keys from both frames
users = pd.merge(users, user4, on="name", how='outer')
users

,name,age,gender,job,height
0,alice,19,F,student,165.0
1,john,26,M,student,180.0
2,eric,22,M,student,175.0
3,paul,58,F,manager,NaN
4,peter,33,M,engineer,NaN
5,julie,44,F,scientist,171.0


### Reshaping by pivoting

In [12]:
# “Unpivots” a DataFrame from wide format to long (stacked) format
staked = pd.melt(users, id_vars="name", var_name="variable", value_name="value")
staked

,name,variable,value
0,alice,age,19
1,john,age,26
2,eric,age,22
3,paul,age,58
4,peter,age,33
5,julie,age,44
6,alice,gender,F
7,john,gender,M
8,eric,gender,M
9,paul,gender,F


In [13]:
# “pivots” a DataFrame from long (stacked) format to wide format,
staked.pivot(index='name', columns='variable', values='value')

variable,age,gender,height,job
name,,,,
alice,19,F,165,student
eric,22,M,175,student
john,26,M,180,student
julie,44,F,171,scientist
paul,58,F,NaN,manager
peter,33,M,NaN,engineer


[<a href="#Pandas:-Data-Manipulation">Back to top</a>]

## 3. Summarizing

In [14]:
users           # print the first 30 and last 30 rows

,name,age,gender,job,height
0,alice,19,F,student,165.0
1,john,26,M,student,180.0
2,eric,22,M,student,175.0
3,paul,58,F,manager,NaN
4,peter,33,M,engineer,NaN
5,julie,44,F,scientist,171.0


In [15]:
type(users)     # DataFrame

pandas.core.frame.DataFrame

In [16]:
users.head()    # print the first 5 rows

,name,age,gender,job,height
0,alice,19,F,student,165.0
1,john,26,M,student,180.0
2,eric,22,M,student,175.0
3,paul,58,F,manager,NaN
4,peter,33,M,engineer,NaN


In [17]:
users.tail()    # print the last 5 rows

,name,age,gender,job,height
1,john,26,M,student,180.0
2,eric,22,M,student,175.0
3,paul,58,F,manager,NaN
4,peter,33,M,engineer,NaN
5,julie,44,F,scientist,171.0


In [18]:
users.index     # "the index" (aka "the labels")

Int64Index([0, 1, 2, 3, 4, 5], dtype='int64')

In [19]:
users.columns   # column names (which is "an index")

Index(['name', 'age', 'gender', 'job', 'height'], dtype='object')

In [20]:
users.dtypes    # data types of each column

name       object
age         int64
gender     object
job        object
height    float64
dtype: object

In [21]:
users.shape     # number of rows and columns

(6, 5)

In [22]:
users.values    # underlying numpy array

array([['alice', 19, 'F', 'student', 165.0],
       ['john', 26, 'M', 'student', 180.0],
       ['eric', 22, 'M', 'student', 175.0],
       ['paul', 58, 'F', 'manager', nan],
       ['peter', 33, 'M', 'engineer', nan],
       ['julie', 44, 'F', 'scientist', 171.0]], dtype=object)

In [23]:
users.info()    # concise summary (includes memory usage as of pandas 0.15.0)

<class 'pandas.core.frame.DataFrame'>
Int64Index: 6 entries, 0 to 5
Data columns (total 5 columns):
name      6 non-null object
age       6 non-null int64
gender    6 non-null object
job       6 non-null object
height    4 non-null float64
dtypes: float64(1), int64(1), object(3)
memory usage: 288.0+ bytes


[<a href="#Pandas:-Data-Manipulation">Back to top</a>]

## 4. Columns selection

In [24]:
users['gender']       # select one column

0    F
1    M
2    M
3    F
4    M
5    F
Name: gender, dtype: object

In [25]:
type(users['gender']) # Series

pandas.core.series.Series

In [26]:
users.gender          # select one column using the DataFrame

0    F
1    M
2    M
3    F
4    M
5    F
Name: gender, dtype: object

In [27]:
# select multiple columns
users[['age', 'gender']]    # select two columns

,age,gender
0,19,F
1,26,M
2,22,M
3,58,F
4,33,M
5,44,F


In [28]:
my_cols = ['age', 'gender'] # or, create a list...
users[my_cols]              # ...and use that list to select columns

,age,gender
0,19,F
1,26,M
2,22,M
3,58,F
4,33,M
5,44,F


In [29]:
type(users[my_cols])        # DataFrame

pandas.core.frame.DataFrame

[<a href="#Pandas:-Data-Manipulation">Back to top</a>]

## 5. Rows selection (basic)

**`iloc`** is strictly integer position based

In [30]:
df = users.copy()
df.iloc[0]  # first row

name        alice
age            19
gender          F
job       student
height        165
Name: 0, dtype: object

In [31]:
df.iloc[0, 0]  # first item of first row

'alice'

In [32]:
df.iloc[0, 0] = 55

for i in range(users.shape[0]):  # users' shape is (6, 5)
    row = df.iloc[i]  # You will get SettingWithCopyWarning
    row.age *= 100    # setting a copy, and not the original frame data.
row

C:\Users\kikim\Anaconda3\envs\tf2.0\lib\site-packages\pandas\core\generic.py:5208: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self[name] = value


name          julie
age            4400
gender            F
job       scientist
height          171
Name: 5, dtype: object

In [33]:
df

,name,age,gender,job,height
0,55,19,F,student,165.0
1,john,26,M,student,180.0
2,eric,22,M,student,175.0
3,paul,58,F,manager,NaN
4,peter,33,M,engineer,NaN
5,julie,44,F,scientist,171.0


**`loc`** supports mixed integer and label based access.

In [34]:
df = users.copy()
df.loc[0]        # first row

name        alice
age            19
gender          F
job       student
height        165
Name: 0, dtype: object

In [35]:
df.loc[0, "age"] # first item("age" column) of first row

19

In [36]:
df.loc[0, "age"] = 55

In [37]:
for i in range(df.shape[0]):
    df.loc[i, "age"] *= 10

df   # df is modified

,name,age,gender,job,height
0,alice,550,F,student,165.0
1,john,260,M,student,180.0
2,eric,220,M,student,175.0
3,paul,580,F,manager,NaN
4,peter,330,M,engineer,NaN
5,julie,440,F,scientist,171.0


[<a href="#Pandas:-Data-Manipulation">Back to top</a>]

## 6. Row selection (filtering)

### Simple logical filtering

In [38]:
users[users.age < 20]         # only show users with age < 20

,name,age,gender,job,height
0,alice,19,F,student,165.0


In [39]:
young_bool = users.age < 20   # or, create a Series of booleans...
young_bool

0     True
1    False
2    False
3    False
4    False
5    False
Name: age, dtype: bool

In [40]:
young = users[young_bool]     # ...and use that Series to filter rows
young

,name,age,gender,job,height
0,alice,19,F,student,165.0


In [41]:
users[users.age < 20].job     # select one column from the filtered results

0    student
Name: job, dtype: object

### Advanced logical filtering

In [42]:
users[users.age < 20][['age', 'job']]            # select multiple columns

,age,job
0,19,student


In [43]:
users[(users.age > 20) & (users.gender == 'M')]  # use multiple conditions

,name,age,gender,job,height
1,john,26,M,student,180.0
2,eric,22,M,student,175.0
4,peter,33,M,engineer,NaN


In [44]:
users[users.job.isin(['student', 'engineer'])]   # filter specific values

,name,age,gender,job,height
0,alice,19,F,student,165.0
1,john,26,M,student,180.0
2,eric,22,M,student,175.0
4,peter,33,M,engineer,NaN


[<a href="#Pandas:-Data-Manipulation">Back to top</a>]

## 7. Sorting

In [45]:
df = users.copy()
df.age.sort_values()     # only works for a Series

0    19
2    22
1    26
4    33
5    44
3    58
Name: age, dtype: int64

In [46]:
df.sort_values(by='age') # sort rows by a specific column

,name,age,gender,job,height
0,alice,19,F,student,165.0
2,eric,22,M,student,175.0
1,john,26,M,student,180.0
4,peter,33,M,engineer,NaN
5,julie,44,F,scientist,171.0
3,paul,58,F,manager,NaN


In [47]:
df.sort_values(by='age', ascending=False)       # use descending order instead

,name,age,gender,job,height
3,paul,58,F,manager,NaN
5,julie,44,F,scientist,171.0
4,peter,33,M,engineer,NaN
1,john,26,M,student,180.0
2,eric,22,M,student,175.0
0,alice,19,F,student,165.0


In [48]:
df.sort_values(by=['job', 'age'])               # sort by multiple columns

,name,age,gender,job,height
4,peter,33,M,engineer,NaN
3,paul,58,F,manager,NaN
5,julie,44,F,scientist,171.0
0,alice,19,F,student,165.0
2,eric,22,M,student,175.0
1,john,26,M,student,180.0


In [49]:
df.sort_values(by=['job', 'age'], inplace=True) # modify df

In [50]:
df

,name,age,gender,job,height
4,peter,33,M,engineer,NaN
3,paul,58,F,manager,NaN
5,julie,44,F,scientist,171.0
0,alice,19,F,student,165.0
2,eric,22,M,student,175.0
1,john,26,M,student,180.0


[<a href="#Pandas:-Data-Manipulation">Back to top</a>]

## 8. Descriptive statistics

### 숫자 형식의 열(column)에 대한 요약 정보 보기

In [51]:
df.describe()

,age,height
count,6.000000,4.000000
mean,33.666667,172.750000
std,14.895189,6.344289
min,19.000000,165.000000
25%,23.000000,169.500000
50%,29.500000,173.000000
75%,41.250000,176.250000
max,58.000000,180.000000


### 모든 열에 대한 요약

In [52]:
df.describe(include='all')

,name,age,gender,job,height
count,6,6.000000,6,6,4.000000
unique,6,NaN,2,4,NaN
top,paul,NaN,M,student,NaN
freq,1,NaN,3,3,NaN
mean,NaN,33.666667,NaN,NaN,172.750000
std,NaN,14.895189,NaN,NaN,6.344289
min,NaN,19.000000,NaN,NaN,165.000000
25%,NaN,23.000000,NaN,NaN,169.500000
50%,NaN,29.500000,NaN,NaN,173.000000
75%,NaN,41.250000,NaN,NaN,176.250000


In [53]:
df.describe(include=['object']) # 데이터 형식 지정

,name,gender,job
count,6,6,6
unique,6,2,4
top,paul,M,student
freq,1,3,3


In [54]:
df.info()   # 포함된 데이터 형식들에 대한 정보

<class 'pandas.core.frame.DataFrame'>
Int64Index: 6 entries, 4 to 1
Data columns (total 5 columns):
name      6 non-null object
age       6 non-null int64
gender    6 non-null object
job       6 non-null object
height    4 non-null float64
dtypes: float64(1), int64(1), object(3)
memory usage: 288.0+ bytes


### 그룹 단위 통계치(groupby)

In [55]:
df.groupby("job").mean()

,age,height
job,,
engineer,33.000000,NaN
manager,58.000000,NaN
scientist,44.000000,171.000000
student,22.333333,173.333333


In [56]:
for grp, data in df.groupby("job"):
    print(grp, data)

engineer     name  age gender       job  height
4  peter   33      M  engineer     NaN
manager    name  age gender      job  height
3  paul   58      F  manager     NaN
scientist     name  age gender        job  height
5  julie   44      F  scientist   171.0
student     name  age gender      job  height
0  alice   19      F  student   165.0
2   eric   22      M  student   175.0
1   john   26      M  student   180.0


[<a href="#Pandas:-Data-Manipulation">Back to top</a>]

## 9. Quality check

### Remove duplicate data

In [57]:
df = users.append(df.iloc[0], ignore_index=True)
df

,name,age,gender,job,height
0,alice,19,F,student,165.0
1,john,26,M,student,180.0
2,eric,22,M,student,175.0
3,paul,58,F,manager,NaN
4,peter,33,M,engineer,NaN
5,julie,44,F,scientist,171.0
6,peter,33,M,engineer,NaN


In [58]:
df.duplicated()   # Series of booleans (어떤 행이 이전 행과 동일하면 True)

0    False
1    False
2    False
3    False
4    False
5    False
6     True
dtype: bool

In [59]:
df.duplicated().sum()                   # 중복된 행의 수

1

In [60]:
df[df.duplicated()]                     # 중복된 행들만 보기

,name,age,gender,job,height
6,peter,33,M,engineer,NaN


In [61]:
df.age.duplicated()                     # 특정 한 열(column)의 중복 체크

0    False
1    False
2    False
3    False
4    False
5    False
6     True
Name: age, dtype: bool

In [62]:
df.duplicated(['age', 'gender']).sum()  # 중복 조사 대상 열들을 열거

1

In [63]:
df = df.drop_duplicates()               # 중복 행들을 삭제
df

,name,age,gender,job,height
0,alice,19,F,student,165.0
1,john,26,M,student,180.0
2,eric,22,M,student,175.0
3,paul,58,F,manager,NaN
4,peter,33,M,engineer,NaN
5,julie,44,F,scientist,171.0


### Missing data

In [64]:
# 누락 값 (Missing values)은 대개 배제된다
df = users.copy()

In [65]:
df.describe(include='all')     # excludes missing values

,name,age,gender,job,height
count,6,6.000000,6,6,4.000000
unique,6,NaN,2,4,NaN
top,paul,NaN,F,student,NaN
freq,1,NaN,3,3,NaN
mean,NaN,33.666667,NaN,NaN,172.750000
std,NaN,14.895189,NaN,NaN,6.344289
min,NaN,19.000000,NaN,NaN,165.000000
25%,NaN,23.000000,NaN,NaN,169.500000
50%,NaN,29.500000,NaN,NaN,173.000000
75%,NaN,41.250000,NaN,NaN,176.250000


**Find missing values in a Series**

In [66]:
df.height.isnull()         # True if NaN, False otherwise

0    False
1    False
2    False
3     True
4     True
5    False
Name: height, dtype: bool

In [67]:
df.height.notnull()        # False if NaN, True otherwise

0     True
1     True
2     True
3    False
4    False
5     True
Name: height, dtype: bool

In [68]:
df[df.height.notnull()]    # only show rows where age is not NaN

,name,age,gender,job,height
0,alice,19,F,student,165.0
1,john,26,M,student,180.0
2,eric,22,M,student,175.0
5,julie,44,F,scientist,171.0


In [69]:
df.height.isnull().sum()   # count the missing values

2

**Find missing values in a DataFrame**

In [70]:
df.isnull()        # DataFrame of booleans

,name,age,gender,job,height
0,False,False,False,False,False
1,False,False,False,False,False
2,False,False,False,False,False
3,False,False,False,False,True
4,False,False,False,False,True
5,False,False,False,False,False


In [71]:
df.isnull().sum()  # calculate the sum of each column

name      0
age       0
gender    0
job       0
height    2
dtype: int64

**Strategy 1: drop missing values**

In [72]:
df.dropna()           # drop a row if ANY values are missing

,name,age,gender,job,height
0,alice,19,F,student,165.0
1,john,26,M,student,180.0
2,eric,22,M,student,175.0
5,julie,44,F,scientist,171.0


In [73]:
df.dropna(how='all')  # drop a row only if ALL values are missing

,name,age,gender,job,height
0,alice,19,F,student,165.0
1,john,26,M,student,180.0
2,eric,22,M,student,175.0
3,paul,58,F,manager,NaN
4,peter,33,M,engineer,NaN
5,julie,44,F,scientist,171.0


**Strategy 2: fill in missing values**

In [74]:
df.height.mean()

172.75

In [75]:
df = users.copy()
df.loc[df.height.isnull(), "height"] = df["height"].mean()
df

,name,age,gender,job,height
0,alice,19,F,student,165.00
1,john,26,M,student,180.00
2,eric,22,M,student,175.00
3,paul,58,F,manager,172.75
4,peter,33,M,engineer,172.75
5,julie,44,F,scientist,171.00


[<a href="#Pandas:-Data-Manipulation">Back to top</a>]

## 10. Rename values

In [76]:
df = users.copy()
df.columns

Index(['name', 'age', 'gender', 'job', 'height'], dtype='object')

In [77]:
df.columns = ['age', 'genre', 'travail', 'nom', 'taille']

In [78]:
df.travail = df.travail.map({ 'student':'etudiant', 'manager':'manager',
                'engineer':'ingenieur', 'scientist':'scientifique'})

In [79]:
assert df.travail.isnull().sum() == 0    # null 이 하나라도 있으면 error 발생시켜라!

AssertionError: 

In [80]:
df['travail'].str.contains("etu|inge")

0    NaN
1    NaN
2    NaN
3    NaN
4    NaN
5    NaN
Name: travail, dtype: object

In [81]:
df

,age,genre,travail,nom,taille
0,alice,19,NaN,student,165.0
1,john,26,NaN,student,180.0
2,eric,22,NaN,student,175.0
3,paul,58,NaN,manager,NaN
4,peter,33,NaN,engineer,NaN
5,julie,44,NaN,scientist,171.0


[<a href="#Pandas:-Data-Manipulation">Back to top</a>]

## 11. Dealing with outliers

In [82]:
size = pd.Series(np.random.normal(loc=175, size=20, scale=10))
# Corrupt the first 3 measures
size[:3] += 600
size

0     782.840072
1     777.757577
2     788.407757
3     166.657866
4     159.386549
5     178.725348
6     185.791420
7     172.273320
8     175.287790
9     183.806973
10    174.076426
11    169.142103
12    179.038673
13    180.529041
14    185.068247
15    176.412646
16    183.443695
17    178.344954
18    169.652457
19    158.467149
dtype: float64

### Based on parametric statistics: use the mean  
* 확률 변수가 정규 분포를 따른다고 가정.
* 표준 편차(standard deviation, SD)의 3 배 밖의 것은 제거한다.
* 샘플 데이터가 1 SD 내에 있을 확률: 68.27%
* 샘플 데이터가 2 SD 내에 있을 확률: 95.45%
* 샘플 데이터가 3 SD 내에 있을 확률: 99.73%

In [83]:
size_outlr_mean = size.copy()
size_outlr_mean[((size - size.mean()).abs() > 3 * size.std())] = size.mean()
size_outlr_mean

0     782.840072
1     777.757577
2     788.407757
3     166.657866
4     159.386549
5     178.725348
6     185.791420
7     172.273320
8     175.287790
9     183.806973
10    174.076426
11    169.142103
12    179.038673
13    180.529041
14    185.068247
15    176.412646
16    183.443695
17    178.344954
18    169.652457
19    158.467149
dtype: float64

### Based on non-parametric statistics: use the median
* 중앙값 절대 편차(Median Absolute Deviation, MAD) 이용 

In [84]:
mad = 1.4826 * np.median(np.abs(size - size.median()))
size_outlr_mad = size.copy()

In [85]:
size_outlr_mad[((size - size.median()).abs() > 3 * mad)] = size.median()
print(size_outlr_mad.mean(), size_outlr_mad.median())

175.5855055199222 178.44005237562396


## 11. File I/O

### csv

In [86]:
import tempfile, os.path
tmpdir = tempfile.gettempdir()
csv_filename = os.path.join(tmpdir, "users.csv")
users.to_csv(csv_filename, index=False)
other = pd.read_csv(csv_filename)

### Read csv from url

In [87]:
url = 'https://docs.terminalfour.com/media/People.csv'
salary = pd.read_csv(url)
salary

,First,Last,Phone,Email
0,Joe,Blogs,8509700,joe.bloggs@terminalfour.com
1,Client,Support,8509777,clientsupport@terminalfour.com


### Excel

**다음과 같이 openpyxl 과 xlrd 설치가 필요할 수 있다**
* conda install openpyxl
* conda install xlrd

In [88]:
# Single worksheet
xls_filename = os.path.join(tmpdir, "users.xlsx")
writer = pd.ExcelWriter(xls_filename)
users.to_excel(xls_filename, sheet_name='users', index=False)

pd.read_excel(xls_filename, sheet_name='users')

ModuleNotFoundError: No module named 'openpyxl'

In [89]:
# Multiple sheets
with pd.ExcelWriter(xls_filename) as writer:
    users.to_excel(writer, sheet_name='users', index=False)
    df.to_excel(writer, sheet_name='salary', index=False)
pd.read_excel(xls_filename, sheet_name='users')

ModuleNotFoundError: No module named 'openpyxl'

In [ ]:
pd.read_excel(xls_filename, sheet_name='salary')

### SQL (SQLite)

In [ ]:
import pandas as pd
import sqlite3
db_filename = os.path.join(tmpdir, "users.db")

**Connect**

In [ ]:
conn = sqlite3.connect(db_filename)

**Creating tables with pandas**

In [ ]:
url = 'http://forge.scilab.org/index.php/p/rdataset/source/tree/master/csv/car/Salaries.csv'
salary = pd.read_csv(url)
salary.to_sql("salary", conn, if_exists="replace")

In [ ]:
salary

**Push modifications**

In [ ]:
cur = conn.cursor()
values = (100, 14000, 5, 'Bachelor', 'N')
cur.execute("insert into salary values (?, ?, ?, ?, ?)", values)
conn.commit()

In [ ]:
salary

**Reading results into a pandas DataFrame**

In [ ]:
salary_sql = pd.read_sql_query("select * from salary;", conn)
print(salary_sql.head())

In [ ]:
pd.read_sql_query("select * from salary;", conn).tail()

In [ ]:
pd.read_sql_query('select * from salary where salary>25000;', conn)

In [ ]:
pd.read_sql_query('select * from salary where experience=16;', conn)

In [ ]:
pd.read_sql_query('select * from salary where education="Master";', conn)

[<a href="#Pandas:-Data-Manipulation">Back to top</a>]